# 📊 Inter-Annotator Agreement — Fleiss' Kappa
**Purpose:** Statistically validate the human-labeled subset before AI evaluation.

| Item | Value |
|---|---|
| Phase 1 (CIB-heavy sample) | 400 suspected buzzers + 100 random → 47 duplicated + deleted 3 rows → **450 unique tweets** |
| Phase 2 (Balanced sample) | 500 suspected non-buzzers → 495 confirmed organic tweets |
| Final Master Ground Truth | **945 tweets** |
| Annotators | 5 independent raters |
| Metric | Fleiss' κ (statsmodels) |
| Ground Truth rule | Majority vote ≥ 3 / 5 |

> The 3 rows deleted in Phase 1: deleted 3 tweets/ rows which is not rated by all 5 annotators.

> **Kappa Paradox note:** Phase 1 alone yields κ ≈ 0.41 despite ~81 % raw agreement,
> because the heavy CIB skew deflates κ mathematically.  
> Adding 495 balanced organic tweets (Phase 2) corrects the marginal distribution → κ ≈ 0.92.


## 0 · Environment Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_FOLDER = "/content/drive/MyDrive/ITS RESEARCH/2026/2026-semantic-cib-detection/DATASET/"
os.makedirs(PROJECT_FOLDER, exist_ok=True)
print(f"✅ Project folder ready: {PROJECT_FOLDER}")

Mounted at /content/drive
✅ Project folder ready: /content/drive/MyDrive/ITS RESEARCH/2026/2026-semantic-cib-detection/DATASET/


In [2]:
# Install / import dependencies
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

print("✅ Libraries loaded.")

✅ Libraries loaded.


## 1 · Diagnostic — Annotator File Audit
Verify each file has unique tweet IDs and report any hidden duplicates before merging.


In [3]:
# File Check on my Gdrive
files_meta = {
    "anno_1.csv": "tweet_id_1",
    "anno_2.csv": "tweet_id_2",
    "anno_3.csv": "tweet_id_3",
    "anno_4.csv": "tweet_id_4",
    "anno_5.csv": "tweet_id_5",
}

print("🔍 ANNOTATOR FILE AUDIT\n" + "="*55)
for fname, id_col in files_meta.items():
    fpath = os.path.join(PROJECT_FOLDER, fname)
    if not os.path.exists(fpath):
        print(f"📭 {fname}: FILE NOT FOUND\n")
        continue
    df = pd.read_csv(fpath).dropna(subset=[id_col])
    total   = len(df)
    unique  = df[id_col].nunique()
    dups    = total - unique
    flag    = "⚠️  HAS DUPLICATES" if dups > 0 else "✅ CLEAN"
    print(f"📄 {fname}")
    print(f"   Total rows    : {total}")
    print(f"   Unique tweets : {unique}")
    print(f"   Duplicates    : {dups}  [{flag}]\n")
print("="*55)

🔍 ANNOTATOR FILE AUDIT
📄 anno_1.csv
   Total rows    : 500
   Unique tweets : 453
   Duplicates    : 47  [⚠️  HAS DUPLICATES]

📄 anno_2.csv
   Total rows    : 500
   Unique tweets : 453
   Duplicates    : 47  [⚠️  HAS DUPLICATES]

📄 anno_3.csv
   Total rows    : 500
   Unique tweets : 453
   Duplicates    : 47  [⚠️  HAS DUPLICATES]

📄 anno_4.csv
   Total rows    : 500
   Unique tweets : 453
   Duplicates    : 47  [⚠️  HAS DUPLICATES]

📄 anno_5.csv
   Total rows    : 500
   Unique tweets : 453
   Duplicates    : 47  [⚠️  HAS DUPLICATES]



## 2 · Phase 1 — Load Buzzer Annotations (N ≈ 450)
Load all five annotator files, rename columns to a common schema, and perform an
inner-join merge so only tweets seen by **all five** annotators are kept.

**Known data-quality fix:** Annotator 3's file contains stray backtick characters
(`` ` ``) in the label column, a known Excel copy-paste artefact. These are
treated as `1` (CIB) and repaired transparently below.


In [4]:
# Load & clean dataset csv from each annotator
def load_annotator(fname, id_col, label_col, rater_name, folder=PROJECT_FOLDER):
    """Load one annotator CSV, rename columns, and clean labels."""
    df = (pd.read_csv(os.path.join(folder, fname))
            [[id_col, label_col]]
            .rename(columns={id_col: 'tweet_id', label_col: rater_name})
            .dropna(subset=['tweet_id']))

    # Fix stray backtick artefact (treat as '1' / CIB)
    backtick_mask = df[rater_name].astype(str).str.strip() == '`'
    n_fixed = backtick_mask.sum()
    if n_fixed > 0:
        print(f"  ⚠️  {rater_name}: {n_fixed} backtick(s) found and corrected → 1 (CIB)")
    df[rater_name] = df[rater_name].astype(str).str.strip().replace('`', '1')

    # Coerce to int; drop any remaining non-numeric rows
    df[rater_name] = pd.to_numeric(df[rater_name], errors='coerce')
    before = len(df)
    df = df.dropna(subset=[rater_name])
    after  = len(df)
    if before != after:
        print(f"  ⚠️  {rater_name}: dropped {before - after} unparseable row(s)")

    df[rater_name] = df[rater_name].astype(int)
    return df


print("📥 Loading Phase 1 (buzzer) annotations...")
s1 = load_annotator("anno_1.csv", "tweet_id_1", "buzzer_1", "S1")
s2 = load_annotator("anno_2.csv", "tweet_id_2", "buzzer_2", "S2")
s3 = load_annotator("anno_3.csv", "tweet_id_3", "buzzer_3", "S3")
s4 = load_annotator("anno_4.csv", "tweet_id_4", "buzzer_4", "S4")
s5 = load_annotator("anno_5.csv", "tweet_id_5", "buzzer_5", "S5")

print("\n🔄 Inner-joining on tweet_id (keeps only tweets seen by ALL 5 annotators)...")
df_iaa_raw = (s1.merge(s2, on='tweet_id')
                .merge(s3, on='tweet_id')
                .merge(s4, on='tweet_id')
                .merge(s5, on='tweet_id'))

print(f"   Rows after merge : {len(df_iaa_raw)}")

📥 Loading Phase 1 (buzzer) annotations...
  ⚠️  S3: 1 backtick(s) found and corrected → 1 (CIB)
  ⚠️  S3: dropped 4 unparseable row(s)

🔄 Inner-joining on tweet_id (keeps only tweets seen by ALL 5 annotators)...
   Rows after merge : 3001


In [5]:
# DEDUPLICATION
# The merge can produce a Cartesian explosion if annotator files contain duplicate tweet_ids. We keep the first occurrence per tweet_id.
before_dedup = len(df_iaa_raw)
df_p1 = df_iaa_raw.drop_duplicates(subset=['tweet_id']).copy()
after_dedup  = len(df_p1)
removed      = before_dedup - after_dedup

print(f"🧹 Deduplication: {before_dedup} rows → {after_dedup} unique tweets")
if removed > 0:
    print(f"   (removed {removed} Cartesian-explosion duplicates)")

# Sanity check — all label values must be 0 or 1
rater_cols = ['S1', 'S2', 'S3', 'S4', 'S5']
for col in rater_cols:
    bad = df_p1[~df_p1[col].isin([0, 1])]
    assert len(bad) == 0, f"❌ Unexpected values in {col}: {bad[col].unique()}"

print("✅ All label values are valid (0 or 1).")
df_p1.head()

🧹 Deduplication: 3001 rows → 450 unique tweets
   (removed 2551 Cartesian-explosion duplicates)
✅ All label values are valid (0 or 1).


,tweet_id,S1,S2,S3,S4,S5
0,1933041453875564996,1,1,1,1,1
1,1926626156456902953,1,1,1,1,1
2,1918900302998307063,1,1,1,1,1
3,1917474342864302212,1,1,1,1,1
4,1917467975923425456,1,1,1,1,1


## 3 · Phase 1 Kappa — The Kappa Paradox
Compute Fleiss' κ on the CIB-heavy Phase 1 sample.
A moderate κ despite high raw agreement is expected, this is the **Kappa Paradox**.


In [6]:
# PHASE 1 KAPPA
counts_p1, _ = aggregate_raters(df_p1[rater_cols])
kappa_p1     = fleiss_kappa(counts_p1, method='fleiss')

df_p1['Sum'] = df_p1[rater_cols].sum(axis=1)
perfect_p1   = df_p1[(df_p1['Sum'] == 5) | (df_p1['Sum'] == 0)]
raw_pct_p1   = len(perfect_p1) / len(df_p1) * 100

cib_count_p1  = (df_p1['Sum'] >= 3).sum()
org_count_p1  = len(df_p1) - cib_count_p1
skew_ratio_p1 = cib_count_p1 / len(df_p1) * 100

print("=" * 60)
print("  PHASE 1 — CIB-HEAVY SAMPLE (Kappa Paradox)")
print("=" * 60)
print(f"  Total unique tweets   : {len(df_p1)}")
print(f"  CIB (majority vote)   : {cib_count_p1}  ({skew_ratio_p1:.1f} %)")
print(f"  Organic               : {org_count_p1}  ({100 - skew_ratio_p1:.1f} %)")
print(f"  Raw perfect agreement : {raw_pct_p1:.1f} %")
print(f"  Fleiss' κ             : {kappa_p1:.4f}   ← moderate despite high raw agreement")
print("=" * 60)
print("  → This is the Kappa Paradox: prevalence skew deflates κ.")

  PHASE 1 — CIB-HEAVY SAMPLE (Kappa Paradox)
  Total unique tweets   : 450
  CIB (majority vote)   : 431  (95.8 %)
  Organic               : 19  (4.2 %)
  Raw perfect agreement : 81.3 %
  Fleiss' κ             : 0.4177   ← moderate despite high raw agreement
  → This is the Kappa Paradox: prevalence skew deflates κ.


## 4 · Phase 2 — Load Organic Annotations (N ≈ 495)
Load the non-buzzer annotation files and apply the same cleaning pipeline.


In [7]:
# PHASE 2 Global Kappa
PREFIX   = "anno_nonbz_"
id_cols  = [f"tweet_id_{i}" for i in range(1, 6)]

print(f"🔍 OVERLAP AUDIT FOR: {PREFIX}*\n" + "="*55)
sets_of_ids = []
for i in range(5):
    fname = f"{PREFIX}{i+1}.csv"
    fpath = os.path.join(PROJECT_FOLDER, fname)
    if os.path.exists(fpath):
        ids = set(pd.read_csv(fpath)[id_cols[i]].dropna().unique())
        sets_of_ids.append(ids)
        print(f"📄 Annotator {i+1}: {len(ids)} unique tweets")
    else:
        print(f"📭 {fname}: FILE NOT FOUND")
        sets_of_ids.append(set())

print("\n🔻 CASCADING INTERSECTION:")
intersection = sets_of_ids[0]
print(f"  S1 alone            : {len(intersection)}")
for i in range(1, 5):
    intersection = intersection.intersection(sets_of_ids[i])
    print(f"  S1 through S{i+1}      : {len(intersection)}")

print("\n🔀 PAIRWISE OVERLAP:")
for i in range(5):
    for j in range(i+1, 5):
        ov = len(sets_of_ids[i].intersection(sets_of_ids[j]))
        flag = "✅" if ov >= 400 else "⚠️  MISMATCH"
        print(f"  S{i+1} & S{j+1}: {ov} tweets  [{flag}]")

🔍 OVERLAP AUDIT FOR: anno_nonbz_*
📄 Annotator 1: 495 unique tweets
📄 Annotator 2: 495 unique tweets
📄 Annotator 3: 495 unique tweets
📄 Annotator 4: 495 unique tweets
📄 Annotator 5: 495 unique tweets

🔻 CASCADING INTERSECTION:
  S1 alone            : 495
  S1 through S2      : 495
  S1 through S3      : 495
  S1 through S4      : 495
  S1 through S5      : 495

🔀 PAIRWISE OVERLAP:
  S1 & S2: 495 tweets  [✅]
  S1 & S3: 495 tweets  [✅]
  S1 & S4: 495 tweets  [✅]
  S1 & S5: 495 tweets  [✅]
  S2 & S3: 495 tweets  [✅]
  S2 & S4: 495 tweets  [✅]
  S2 & S5: 495 tweets  [✅]
  S3 & S4: 495 tweets  [✅]
  S3 & S5: 495 tweets  [✅]
  S4 & S5: 495 tweets  [✅]


In [8]:
# Load PHASE 2 Global Kappa
print("📥 Loading Phase 2 (organic / non-buzzer) annotations...")
n1 = load_annotator("anno_nonbz_1.csv", "tweet_id_1", "buzzer_1", "S1")
n2 = load_annotator("anno_nonbz_2.csv", "tweet_id_2", "buzzer_2", "S2")
n3 = load_annotator("anno_nonbz_3.csv", "tweet_id_3", "buzzer_3", "S3")
n4 = load_annotator("anno_nonbz_4.csv", "tweet_id_4", "buzzer_4", "S4")
n5 = load_annotator("anno_nonbz_5.csv", "tweet_id_5", "buzzer_5", "S5")

df_neg_raw = (n1.merge(n2, on='tweet_id')
                .merge(n3, on='tweet_id')
                .merge(n4, on='tweet_id')
                .merge(n5, on='tweet_id'))

# Dedup, clean
df_p2 = df_neg_raw.drop_duplicates(subset=['tweet_id']).copy()
df_p2 = df_p2[~df_p2.astype(str).isin(['`']).any(axis=1)]   # remove residual backticks
for col in rater_cols:
    df_p2[col] = pd.to_numeric(df_p2[col], errors='coerce')
df_p2 = df_p2.dropna(subset=rater_cols)
for col in rater_cols:
    df_p2[col] = df_p2[col].astype(int)

dup_count = df_neg_raw.duplicated(subset=['tweet_id']).sum()
print(f"   Phase 2 rows after merge      : {len(df_neg_raw)}")
print(f"   Cartesian duplicates removed  : {dup_count}")
print(f"   Phase 2 unique tweets (df_p2) : {len(df_p2)}")
df_p2.head()

📥 Loading Phase 2 (organic / non-buzzer) annotations...
   Phase 2 rows after merge      : 650
   Cartesian duplicates removed  : 155
   Phase 2 unique tweets (df_p2) : 495


,tweet_id,S1,S2,S3,S4,S5
0,1975948493371146389,0,0,0,0,0
1,1918971274803462523,0,0,0,0,0
2,1918679321180594605,0,0,0,0,0
3,1890422946205782315,0,0,0,0,0
4,1890418203760767251,0,0,0,0,0


## 5 · Build Master Ground Truth & Global Kappa
Concatenate Phase 1 + Phase 2, compute the corrected Global κ, and establish
the binary Ground Truth via majority vote (≥ 3 / 5 annotators).


In [9]:
# CONCATENATE P1 + P2
df_master = pd.concat(
    [df_p1[['tweet_id'] + rater_cols],
     df_p2[['tweet_id'] + rater_cols]],
    ignore_index=True
)

# Integrity check, no tweet_id should appear in both phases
overlap_ids = set(df_p1['tweet_id']).intersection(set(df_p2['tweet_id']))
assert len(overlap_ids) == 0, f"❌ {len(overlap_ids)} tweet_ids appear in BOTH phases!"

print(f"✅ No tweet_id overlap between Phase 1 and Phase 2.")
print(f"🔗 Master dataset: {len(df_p1)} (P1) + {len(df_p2)} (P2) = {len(df_master)} total tweets")

✅ No tweet_id overlap between Phase 1 and Phase 2.
🔗 Master dataset: 450 (P1) + 495 (P2) = 945 total tweets


In [10]:
# Calculation GLobal Kappa
counts_global, _ = aggregate_raters(df_master[rater_cols])
kappa_global     = fleiss_kappa(counts_global, method='fleiss')

df_master['Sum'] = df_master[rater_cols].sum(axis=1)
perfect_global   = df_master[(df_master['Sum'] == 5) | (df_master['Sum'] == 0)]
raw_pct_global   = len(perfect_global) / len(df_master) * 100

# Ground Truth: majority vote
df_master['Ground_Truth'] = (df_master['Sum'] >= 3).astype(int)
cib_final = df_master['Ground_Truth'].sum()
org_final = len(df_master) - cib_final

print("=" * 60)
print("  GLOBAL KAPPA: BALANCED MASTER DATASET")
print("=" * 60)
print(f"  Total tweets          : {len(df_master)}")
print(f"  CIB (Ground Truth=1)  : {cib_final}  ({cib_final/len(df_master)*100:.1f} %)")
print(f"  Organic (GT=0)        : {org_final}  ({org_final/len(df_master)*100:.1f} %)")
print(f"  Raw perfect agreement : {raw_pct_global:.1f} %")
print(f"  Fleiss' κ (Global)    : {kappa_global:.4f}  ← Global Kappa after balancing")
print("=" * 60)

# SUMMARY
print("\n  PHASE COMPARISON:")
print(f"  {'Phase':<30} {'N':>6}  {'κ':>8}  {'Raw Agree':>10}  {'CIB%':>7}")
print("  " + "-"*60)
print(f"  {'Phase 1 (CIB-heavy)':<30} {len(df_p1):>6}  {kappa_p1:>8.4f}  {raw_pct_p1:>9.1f}%  {cib_count_p1/len(df_p1)*100:>6.1f}%")
print(f"  {'Global (P1 + P2 balanced)':<30} {len(df_master):>6}  {kappa_global:>8.4f}  {raw_pct_global:>9.1f}%  {cib_final/len(df_master)*100:>6.1f}%")

  GLOBAL KAPPA: BALANCED MASTER DATASET
  Total tweets          : 945
  CIB (Ground Truth=1)  : 431  (45.6 %)
  Organic (GT=0)        : 514  (54.4 %)
  Raw perfect agreement : 91.1 %
  Fleiss' κ (Global)    : 0.9201  ← Global Kappa after balancing

  PHASE COMPARISON:
  Phase                               N         κ   Raw Agree     CIB%
  ------------------------------------------------------------
  Phase 1 (CIB-heavy)               450    0.4177       81.3%    95.8%
  Global (P1 + P2 balanced)         945    0.9201       91.1%    45.6%


## 6 · Save Outputs

In [11]:
# SAVE
# Phase 1 ground truth (positive-skewed — for Kappa Paradox documentation)
path_p1 = os.path.join(PROJECT_FOLDER, "positive-skewed-ground-truth.csv")
df_p1_save = df_p1[['tweet_id'] + rater_cols + ['Sum']].copy()
df_p1_save['Ground_Truth'] = (df_p1_save['Sum'] >= 3).astype(int)
df_p1_save.to_csv(path_p1, index=False)
print(f"✅ Phase 1 (skewed) saved  : {path_p1}  ({len(df_p1_save)} rows)")

# Final master ground truth (balanced — used for AI evaluation)
path_master = os.path.join(PROJECT_FOLDER, "df-final-ground-truth.csv")
df_master.to_csv(path_master, index=False)
print(f"✅ Master ground truth saved: {path_master}  ({len(df_master)} rows)")

# Sanity assertions before saving
assert len(df_master) == len(df_p1) + len(df_p2), "Row count mismatch!"
assert df_master['Ground_Truth'].isin([0,1]).all(), "Ground_Truth has unexpected values!"
assert df_master['tweet_id'].nunique() == len(df_master), "Duplicate tweet_ids in master!"
print("\n✅ All sanity checks passed.")

✅ Phase 1 (skewed) saved  : /content/drive/MyDrive/ITS RESEARCH/2026/2026-semantic-cib-detection/DATASET/positive-skewed-ground-truth.csv  (450 rows)
✅ Master ground truth saved: /content/drive/MyDrive/ITS RESEARCH/2026/2026-semantic-cib-detection/DATASET/df-final-ground-truth.csv  (945 rows)

✅ All sanity checks passed.
